# 01_Data_Exploration

## Precipitation Downscaling Khulna

Objective:
- Explore input datasets
- Check raster properties
- Verify CRS and resolution
- Visualize spatial datasets

Datasets:
- CHIRPS precipitation
- ERA5 precipitation
- NDVI
- LST
- Land variables
- Distance from Sea
- Khulna boundary


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

import rasterio
from rasterio.plot import show

import geopandas as gpd

import matplotlib.pyplot as plt

import yaml

In [3]:
PROJECT_ROOT = Path(
    r"E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna"
)

print(PROJECT_ROOT)

E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna


In [4]:
with open(PROJECT_ROOT / "environment.yml",
          "r",
          encoding="utf-8") as file:
    
    config = yaml.safe_load(file)


print("Configuration Loaded Successfully")

Configuration Loaded Successfully


In [5]:
# ==========================================================
# Define Project Paths
# ==========================================================

# Main folders
RAW_DATA = PROJECT_ROOT / config["paths"]["raw_data"]
PROCESSED_DATA = PROJECT_ROOT / config["paths"]["processed_data"]
INTERIM_DATA = PROJECT_ROOT / config["paths"]["interim_data"]
OUTPUT_DATA = PROJECT_ROOT / config["paths"]["output_data"]

# Results
FIGURES = PROJECT_ROOT / config["results"]["figures"]
TABLES = PROJECT_ROOT / config["results"]["tables"]
MAPS = PROJECT_ROOT / config["results"]["maps"]

# ==========================================================
# Input Datasets
# ==========================================================

BMD = PROJECT_ROOT / config["paths"]["bmd"]
CCS = PROJECT_ROOT / config["paths"]["ccs"]
CDR = PROJECT_ROOT / config["paths"]["cdr"]

CHIRPS = PROJECT_ROOT / config["paths"]["chirps"]

DISTANCE_SEA = PROJECT_ROOT / config["paths"]["distance_sea"]

ERA5 = PROJECT_ROOT / config["paths"]["era5"]

GSMAP = PROJECT_ROOT / config["paths"]["gsmap"]
GSMAP_MVK = PROJECT_ROOT / config["paths"]["gsmap_mvk"]

IMERG = PROJECT_ROOT / config["paths"]["imerg"]

BOUNDARY = PROJECT_ROOT / config["paths"]["boundary"]

LAND_VARIABLE = PROJECT_ROOT / config["paths"]["land_variable"]

LST = PROJECT_ROOT / config["paths"]["lst"]

NDVI = PROJECT_ROOT / config["paths"]["ndvi"]

PDIR = PROJECT_ROOT / config["paths"]["pdir"]

PERSIANN = PROJECT_ROOT / config["paths"]["persiann"]

print("All project paths loaded successfully.")

All project paths loaded successfully.


In [6]:
# ==========================================================
# Check All Dataset Paths
# ==========================================================

datasets = {
    "BMD": BMD,
    "CCS": CCS,
    "CDR": CDR,
    "CHIRPS": CHIRPS,
    "Distance Sea": DISTANCE_SEA,
    "ERA5": ERA5,
    "GSMaP Gauge": GSMAP,
    "GSMaP MVK": GSMAP_MVK,
    "IMERG": IMERG,
    "Khulna Boundary": BOUNDARY,
    "Land Variable": LAND_VARIABLE,
    "LST": LST,
    "NDVI": NDVI,
    "PDIR": PDIR,
    "PERSIANN": PERSIANN,
}

print("=" * 90)
print("DATASET AVAILABILITY")
print("=" * 90)

for name, path in datasets.items():
    status = "✓ FOUND" if path.exists() else "✗ NOT FOUND"
    print(f"{status:<12} | {name:<20} | {path}")

print("=" * 90)

DATASET AVAILABILITY
✓ FOUND      | BMD                  | E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\Raw\BMD_Data
✓ FOUND      | CCS                  | E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\Raw\CCS
✓ FOUND      | CDR                  | E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\Raw\CDR
✓ FOUND      | CHIRPS               | E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\Raw\CHIRPS_TIFF_2017_2022
✓ FOUND      | Distance Sea         | E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\Raw\Distance_from_Sea
✓ FOUND      | ERA5                 | E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\Raw\ERA5_TIFF
✓ FOUND      | GSMaP Gauge          | E:\Geospatial\Precipitation Downscaling\Precipitation-Downscaling-Khulna\data\Raw\GSMaP_Gauge_v7
✓ FOUND      | GSMaP MVK            | E:\Geospatial\Preci

In [8]:
# ==========================================================
# Count Files in Each Dataset
# ==========================================================

for name, path in datasets.items():
    tif_files = list(path.glob("*.tif"))
    shp_files = list(path.glob("*.shp"))
    csv_files = list(path.glob("*.csv"))

    total = len(tif_files) + len(shp_files) + len(csv_files)

    print(f"{name:<20} : {total:>4} files")

BMD                  :    1 files
CCS                  :   72 files
CDR                  :   72 files
CHIRPS               :   72 files
Distance Sea         :    8 files
ERA5                 :   72 files
GSMaP Gauge          :   72 files
GSMaP MVK            :   72 files
IMERG                :   72 files
Khulna Boundary      :    1 files
Land Variable        :    3 files
LST                  :   72 files
NDVI                 :   72 files
PDIR                 :   72 files
PERSIANN             :   72 files


In [13]:
# ==========================================================
# Show All Files in Every Dataset
# ==========================================================

for name, path in datasets.items():

    print("\n" + "=" * 80)
    print(f"Dataset: {name}")
    print("=" * 80)

    files = sorted(path.iterdir())

    for file in files:
        print(file.name)

    print(f"\nTotal Files: {len(files)}")


Dataset: BMD
BMD.cpg
BMD.dbf
BMD.prj
BMD.sbn
BMD.sbx
BMD.shp
BMD.shx
CL503.xlsx
CL504.xlsx
CL509.xlsx
CL510.xlsx
CL515.xlsx
CL517.xlsx
Khulna_Monthly_Rainfall_2017_2022_with_LatLong.xls

Total Files: 14

Dataset: CCS
2017_01.tif
2017_02.tif
2017_03.tif
2017_04.tif
2017_05.tif
2017_06.tif
2017_07.tif
2017_08.tif
2017_09.tif
2017_10.tif
2017_11.tif
2017_12.tif
2018_01.tif
2018_02.tif
2018_03.tif
2018_04.tif
2018_05.tif
2018_06.tif
2018_07.tif
2018_08.tif
2018_09.tif
2018_10.tif
2018_11.tif
2018_12.tif
2019_01.tif
2019_02.tif
2019_03.tif
2019_04.tif
2019_05.tif
2019_06.tif
2019_07.tif
2019_08.tif
2019_09.tif
2019_10.tif
2019_11.tif
2019_12.tif
2020_01.tif
2020_02.tif
2020_03.tif
2020_04.tif
2020_05.tif
2020_06.tif
2020_07.tif
2020_08.tif
2020_09.tif
2020_10.tif
2020_11.tif
2020_12.tif
2021_01.tif
2021_02.tif
2021_03.tif
2021_04.tif
2021_05.tif
2021_06.tif
2021_07.tif
2021_08.tif
2021_09.tif
2021_10.tif
2021_11.tif
2021_12.tif
2022_01.tif
2022_02.tif
2022_03.tif
2022_04.tif
2022_05.tif
20

In [14]:
# ==========================================================
# Show All Files with Extension
# ==========================================================

for name, path in datasets.items():

    print("\n" + "=" * 80)
    print(f"Dataset: {name}")
    print("=" * 80)

    files = sorted(path.iterdir())

    for i, file in enumerate(files, start=1):
        print(f"{i:3d}. {file.name}")

    print(f"\nTotal Files: {len(files)}")


Dataset: BMD
  1. BMD.cpg
  2. BMD.dbf
  3. BMD.prj
  4. BMD.sbn
  5. BMD.sbx
  6. BMD.shp
  7. BMD.shx
  8. CL503.xlsx
  9. CL504.xlsx
 10. CL509.xlsx
 11. CL510.xlsx
 12. CL515.xlsx
 13. CL517.xlsx
 14. Khulna_Monthly_Rainfall_2017_2022_with_LatLong.xls

Total Files: 14

Dataset: CCS
  1. 2017_01.tif
  2. 2017_02.tif
  3. 2017_03.tif
  4. 2017_04.tif
  5. 2017_05.tif
  6. 2017_06.tif
  7. 2017_07.tif
  8. 2017_08.tif
  9. 2017_09.tif
 10. 2017_10.tif
 11. 2017_11.tif
 12. 2017_12.tif
 13. 2018_01.tif
 14. 2018_02.tif
 15. 2018_03.tif
 16. 2018_04.tif
 17. 2018_05.tif
 18. 2018_06.tif
 19. 2018_07.tif
 20. 2018_08.tif
 21. 2018_09.tif
 22. 2018_10.tif
 23. 2018_11.tif
 24. 2018_12.tif
 25. 2019_01.tif
 26. 2019_02.tif
 27. 2019_03.tif
 28. 2019_04.tif
 29. 2019_05.tif
 30. 2019_06.tif
 31. 2019_07.tif
 32. 2019_08.tif
 33. 2019_09.tif
 34. 2019_10.tif
 35. 2019_11.tif
 36. 2019_12.tif
 37. 2020_01.tif
 38. 2020_02.tif
 39. 2020_03.tif
 40. 2020_04.tif
 41. 2020_05.tif
 42. 2020_06.ti

In [19]:
from pathlib import Path
import pandas as pd

# ==========================================================
# Scan all files from all datasets
# ==========================================================

records = []

for dataset_name, folder_path in datasets.items():

    for file_path in sorted(folder_path.rglob("*")):

        if file_path.is_file():

            extension = file_path.suffix.lower()

            if extension in [".shp", ".shx", ".dbf", ".prj", ".cpg", ".sbn", ".sbx"]:
                file_group = "Shapefile"

            elif extension in [".xlsx", ".xls", ".csv"]:
                file_group = "Excel / CSV"

            elif extension in [".tif", ".tiff"]:
                file_group = "Raster TIFF"

            else:
                file_group = "Other"

            records.append(
                {
                    "Dataset": dataset_name,
                    "File Name": file_path.name,
                    "File Type": file_group,
                    "Extension": extension,
                    "Folder": str(file_path.parent),
                    "Full Path": str(file_path),
                    "Size MB": round(file_path.stat().st_size / (1024 * 1024), 3),
                }
            )

all_files_df = pd.DataFrame(records)

print(f"Total files found: {len(all_files_df)}")

Total files found: 862


In [20]:
shapefile_df = all_files_df[
    all_files_df["Extension"] == ".shp"
].reset_index(drop=True)

shapefile_df

,Dataset,File Name,File Type,Extension,Folder,Full Path,Size MB
0,BMD,BMD.shp,Shapefile,.shp,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.012
1,Distance Sea,Coastline_UTM.shp,Shapefile,.shp,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.027
2,Distance Sea,Distance_from_Sea.shp,Shapefile,.shp,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.001
3,Distance Sea,grid_centroid_UTM.shp,Shapefile,.shp,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.001
4,Khulna Boundary,Khulna.shp,Shapefile,.shp,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.104


In [21]:
shapefile_components_df = all_files_df[
    all_files_df["File Type"] == "Shapefile"
].reset_index(drop=True)

shapefile_components_df

,Dataset,File Name,File Type,Extension,Folder,Full Path,Size MB
0,BMD,BMD.cpg,Shapefile,.cpg,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.000
1,BMD,BMD.dbf,Shapefile,.dbf,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.243
2,BMD,BMD.prj,Shapefile,.prj,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.000
3,BMD,BMD.sbn,Shapefile,.sbn,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.004
4,BMD,BMD.sbx,Shapefile,.sbx,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.000
5,BMD,BMD.shp,Shapefile,.shp,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.012
6,BMD,BMD.shx,Shapefile,.shx,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.003
7,Distance Sea,Coastline_UTM.cpg,Shapefile,.cpg,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.000
8,Distance Sea,Coastline_UTM.dbf,Shapefile,.dbf,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.002
9,Distance Sea,Coastline_UTM.prj,Shapefile,.prj,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.000


In [22]:
excel_csv_df = all_files_df[
    all_files_df["File Type"] == "Excel / CSV"
].reset_index(drop=True)

excel_csv_df

,Dataset,File Name,File Type,Extension,Folder,Full Path,Size MB
0,BMD,CL503.xlsx,Excel / CSV,.xlsx,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.503
1,BMD,CL504.xlsx,Excel / CSV,.xlsx,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.523
2,BMD,CL509.xlsx,Excel / CSV,.xlsx,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.658
3,BMD,CL510.xlsx,Excel / CSV,.xlsx,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.878
4,BMD,CL515.xlsx,Excel / CSV,.xlsx,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.692
5,BMD,CL517.xlsx,Excel / CSV,.xlsx,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.187
6,BMD,Khulna_Monthly_Rainfall_2017_2022_with_LatLong...,Excel / CSV,.xls,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.070


In [23]:
tiff_df = all_files_df[
    all_files_df["File Type"] == "Raster TIFF"
].reset_index(drop=True)

tiff_df

,Dataset,File Name,File Type,Extension,Folder,Full Path,Size MB
0,CCS,2017_01.tif,Raster TIFF,.tif,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.002
1,CCS,2017_02.tif,Raster TIFF,.tif,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.002
2,CCS,2017_03.tif,Raster TIFF,.tif,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.002
3,CCS,2017_04.tif,Raster TIFF,.tif,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.002
4,CCS,2017_05.tif,Raster TIFF,.tif,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.002
...,...,...,...,...,...,...,...
795,PERSIANN,2022_08.tif,Raster TIFF,.tif,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.001
796,PERSIANN,2022_09.tif,Raster TIFF,.tif,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.001
797,PERSIANN,2022_10.tif,Raster TIFF,.tif,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.001
798,PERSIANN,2022_11.tif,Raster TIFF,.tif,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.001


In [24]:
other_files_df = all_files_df[
    all_files_df["File Type"] == "Other"
].reset_index(drop=True)

other_files_df

,Dataset,File Name,File Type,Extension,Folder,Full Path,Size MB
0,Distance Sea,Coastline_UTM.shp.xml,Other,.xml,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.005
1,Distance Sea,Distance_from_Sea.shp.xml,Other,.xml,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.007
2,Distance Sea,Distance_Sea.tfw,Other,.tfw,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.000
3,Distance Sea,Distance_Sea.tif.aux.xml,Other,.xml,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.001
4,Distance Sea,Distance_Sea.tif.ovr,Other,.ovr,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.001
5,Distance Sea,Distance_Sea.tif.xml,Other,.xml,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.006
6,Distance Sea,Distance_Sea_clip.tfw,Other,.tfw,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.000
7,Distance Sea,Distance_Sea_clip.tif.aux.xml,Other,.xml,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.001
8,Distance Sea,Distance_Sea_clip.tif.xml,Other,.xml,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.001
9,Distance Sea,Distance_Sea_clip2.tfw,Other,.tfw,E:\Geospatial\Precipitation Downscaling\Precip...,E:\Geospatial\Precipitation Downscaling\Precip...,0.000


In [25]:
file_summary_df = (
    all_files_df
    .groupby(["Dataset", "File Type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

file_summary_df

File Type,Dataset,Excel / CSV,Other,Raster TIFF,Shapefile
0,BMD,7,0,0,7
1,CCS,0,0,72,0
2,CDR,0,0,72,0
3,CHIRPS,0,0,72,0
4,Distance Sea,0,19,5,21
5,ERA5,0,0,72,0
6,GSMaP Gauge,0,0,72,0
7,GSMaP MVK,0,0,72,0
8,IMERG,0,0,72,0
9,Khulna Boundary,0,0,0,7


In [26]:
dataset_summary_df = (
    all_files_df
    .groupby("Dataset")
    .agg(
        Total_Files=("File Name", "count"),
        TIFF_Files=("Extension", lambda x: x.isin([".tif", ".tiff"]).sum()),
        SHP_Files=("Extension", lambda x: (x == ".shp").sum()),
        Excel_Files=("Extension", lambda x: x.isin([".xlsx", ".xls", ".csv"]).sum()),
        Total_Size_MB=("Size MB", "sum"),
    )
    .reset_index()
)

dataset_summary_df

,Dataset,Total_Files,TIFF_Files,SHP_Files,Excel_Files,Total_Size_MB
0,BMD,14,0,1,7,3.773
1,CCS,72,72,0,0,0.144
2,CDR,72,72,0,0,0.072
3,CHIRPS,72,72,0,0,0.216
4,Distance Sea,45,5,3,0,0.270
5,ERA5,72,72,0,0,0.144
6,GSMaP Gauge,72,72,0,0,0.144
7,GSMaP MVK,72,72,0,0,0.144
8,IMERG,72,72,0,0,0.144
9,Khulna Boundary,7,0,1,0,0.106


In [27]:
import rasterio
import pandas as pd

def raster_metadata(file_path, dataset):

    with rasterio.open(file_path) as src:

        return {
            "Dataset": dataset,
            "File": file_path.name,
            "CRS": str(src.crs),
            "Width": src.width,
            "Height": src.height,
            "Bands": src.count,
            "Resolution_X": src.res[0],
            "Resolution_Y": src.res[1],
            "DataType": src.dtypes[0],
            "NoData": src.nodata,
            "Left": round(src.bounds.left, 2),
            "Bottom": round(src.bounds.bottom, 2),
            "Right": round(src.bounds.right, 2),
            "Top": round(src.bounds.top, 2),
        }

In [28]:
raster_datasets = {
    "CCS": CCS,
    "CDR": CDR,
    "CHIRPS": CHIRPS,
    "ERA5": ERA5,
    "GSMaP Gauge": GSMAP,
    "GSMaP MVK": GSMAP_MVK,
    "IMERG": IMERG,
    "LST": LST,
    "NDVI": NDVI,
    "PDIR": PDIR,
    "PERSIANN": PERSIANN,
    "Land Variable": LAND_VARIABLE,
    "Distance Sea": DISTANCE_SEA,
}

In [29]:
metadata = []

for dataset, folder in raster_datasets.items():

    tif_files = sorted(folder.glob("*.tif"))

    for tif in tif_files:

        metadata.append(
            raster_metadata(tif, dataset)
        )

metadata_df = pd.DataFrame(metadata)

metadata_df

,Dataset,File,CRS,Width,Height,Bands,Resolution_X,Resolution_Y,DataType,NoData,Left,Bottom,Right,Top
0,CCS,2017_01.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,1,0.040000,0.040000,int16,-9.900000e+01,89.20,21.64,89.80,23.08
1,CCS,2017_02.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,1,0.040000,0.040000,int16,-9.900000e+01,89.20,21.64,89.80,23.08
2,CCS,2017_03.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,1,0.040000,0.040000,int16,-9.900000e+01,89.20,21.64,89.80,23.08
3,CCS,2017_04.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,1,0.040000,0.040000,int16,-9.900000e+01,89.20,21.64,89.80,23.08
4,CCS,2017_05.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,1,0.040000,0.040000,int16,-9.900000e+01,89.20,21.64,89.80,23.08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,Distance Sea,Distance_Sea.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,31,1,0.050000,0.050000,float32,-3.402823e+38,89.15,21.53,89.90,23.08
796,Distance Sea,Distance_Sea_clip.tif,"PROJCS[""WGS_1984_UTM_Zone_46N"",GEOGCS[""WGS 84""...",12,30,1,5394.264043,5394.264043,float32,-3.402823e+38,106539.95,2389895.55,171271.12,2551723.47
797,Distance Sea,Distance_Sea_clip2.tif,"PROJCS[""WGS_1984_UTM_Zone_46N"",GEOGCS[""WGS 84""...",12,29,1,5394.264043,5394.264043,float32,-3.402823e+38,106539.95,2395286.60,171271.12,2551720.26
798,Distance Sea,Distance_Sea_km.tif,"PROJCS[""WGS_1984_UTM_Zone_46N"",GEOGCS[""WGS 84""...",12,30,1,5394.264043,5394.264043,float32,-3.402823e+38,106539.95,2389895.55,171271.12,2551723.47


In [30]:
metadata_df.groupby("Dataset").agg(
    Files=("File", "count"),
    CRS=("CRS", "first"),
    Width=("Width", "first"),
    Height=("Height", "first"),
    Bands=("Bands", "first"),
    Resolution_X=("Resolution_X", "first"),
    Resolution_Y=("Resolution_Y", "first"),
)

,Files,CRS,Width,Height,Bands,Resolution_X,Resolution_Y
Dataset,,,,,,,
CCS,72,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,1,0.040000,0.040000
CDR,72,"LOCAL_CS[""unnamed"",UNIT[""metre"",1,AUTHORITY[""E...",5,8,1,0.250000,0.250000
CHIRPS,72,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",12,28,1,0.050000,0.050000
Distance Sea,5,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,31,1,0.050000,0.050000
ERA5,72,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",4,7,1,0.250001,0.250001
GSMaP Gauge,72,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",6,15,1,0.100000,0.100000
GSMaP MVK,72,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",6,15,1,0.100000,0.100000
IMERG,72,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",6,15,1,0.100000,0.100000
LST,72,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",59,151,1,0.008983,0.008983


In [31]:
metadata_df["CRS"].value_counts()

CRS
GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]                                                                                                                                                                                                                                                                                 721
LOCAL_CS["unnamed",UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]                                                                                                                                                                                                                                                                                                                                                                                                      73
PR

In [32]:
metadata_df[
    [
        "Dataset",
        "File",
        "Resolution_X",
        "Resolution_Y"
    ]
]

,Dataset,File,Resolution_X,Resolution_Y
0,CCS,2017_01.tif,0.040000,0.040000
1,CCS,2017_02.tif,0.040000,0.040000
2,CCS,2017_03.tif,0.040000,0.040000
3,CCS,2017_04.tif,0.040000,0.040000
4,CCS,2017_05.tif,0.040000,0.040000
...,...,...,...,...
795,Distance Sea,Distance_Sea.tif,0.050000,0.050000
796,Distance Sea,Distance_Sea_clip.tif,5394.264043,5394.264043
797,Distance Sea,Distance_Sea_clip2.tif,5394.264043,5394.264043
798,Distance Sea,Distance_Sea_km.tif,5394.264043,5394.264043


In [33]:
metadata_df[
    [
        "Dataset",
        "Width",
        "Height"
    ]
].drop_duplicates()

,Dataset,Width,Height
0,CCS,15,36
72,CDR,5,8
144,CHIRPS,12,28
216,ERA5,4,7
288,GSMaP Gauge,6,15
360,GSMaP MVK,6,15
367,GSMaP MVK,2,3
432,IMERG,6,15
504,LST,59,151
576,NDVI,59,151


In [34]:
metadata_df[
    [
        "Dataset",
        "NoData"
    ]
].drop_duplicates()

,Dataset,NoData
0,CCS,-9.900000e+01
72,CDR,-9.900000e+01
144,CHIRPS,NaN
216,ERA5,NaN
288,GSMaP Gauge,NaN
360,GSMaP MVK,NaN
432,IMERG,NaN
504,LST,NaN
576,NDVI,NaN
648,PDIR,-9.900000e+01


In [35]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

In [36]:
raster_datasets = {
    "CCS": CCS,
    "CDR": CDR,
    "CHIRPS": CHIRPS,
    "ERA5": ERA5,
    "GSMaP Gauge": GSMAP,
    "GSMaP MVK": GSMAP_MVK,
    "IMERG": IMERG,
    "LST": LST,
    "NDVI": NDVI,
    "PDIR": PDIR,
    "PERSIANN": PERSIANN,
    "Land Variable": LAND_VARIABLE,
    "Distance Sea": DISTANCE_SEA,
}

In [37]:
def get_raster_metadata(dataset_name, file_path):
    with rasterio.open(file_path) as src:
        array = src.read(1, masked=True)

        valid_values = array.compressed()

        if valid_values.size > 0:
            minimum = float(valid_values.min())
            maximum = float(valid_values.max())
            mean = float(valid_values.mean())
            standard_deviation = float(valid_values.std())
        else:
            minimum = np.nan
            maximum = np.nan
            mean = np.nan
            standard_deviation = np.nan

        return {
            "Dataset": dataset_name,
            "File_Name": file_path.name,
            "CRS": str(src.crs),
            "Width": src.width,
            "Height": src.height,
            "Bands": src.count,
            "Resolution_X": abs(src.res[0]),
            "Resolution_Y": abs(src.res[1]),
            "Data_Type": src.dtypes[0],
            "NoData": src.nodata,
            "Minimum": minimum,
            "Maximum": maximum,
            "Mean": mean,
            "Std_Dev": standard_deviation,
            "Left": src.bounds.left,
            "Bottom": src.bounds.bottom,
            "Right": src.bounds.right,
            "Top": src.bounds.top,
        }

In [38]:
metadata_records = []
failed_files = []

for dataset_name, folder_path in raster_datasets.items():
    tif_files = sorted(
        list(folder_path.rglob("*.tif")) +
        list(folder_path.rglob("*.tiff"))
    )

    for tif_file in tif_files:
        try:
            metadata_records.append(
                get_raster_metadata(dataset_name, tif_file)
            )
        except Exception as error:
            failed_files.append({
                "Dataset": dataset_name,
                "File_Name": tif_file.name,
                "Error": str(error),
            })

metadata_df = pd.DataFrame(metadata_records)
failed_df = pd.DataFrame(failed_files)

print("Raster files checked:", len(metadata_df))
print("Failed files:", len(failed_df))

Raster files checked: 800
Failed files: 0


In [39]:
metadata_df

,Dataset,File_Name,CRS,Width,Height,Bands,Resolution_X,Resolution_Y,Data_Type,NoData,Minimum,Maximum,Mean,Std_Dev,Left,Bottom,Right,Top
0,CCS,2017_01.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,1,0.040000,0.040000,int16,-9.900000e+01,0.0,3.000000,0.391837,0.535985,89.200000,2.164000e+01,89.800000,2.308000e+01
1,CCS,2017_02.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,1,0.040000,0.040000,int16,-9.900000e+01,0.0,0.000000,0.000000,0.000000,89.200000,2.164000e+01,89.800000,2.308000e+01
2,CCS,2017_03.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,1,0.040000,0.040000,int16,-9.900000e+01,0.0,26.000000,4.253061,4.350140,89.200000,2.164000e+01,89.800000,2.308000e+01
3,CCS,2017_04.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,1,0.040000,0.040000,int16,-9.900000e+01,18.0,113.000000,42.048980,18.537487,89.200000,2.164000e+01,89.800000,2.308000e+01
4,CCS,2017_05.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,1,0.040000,0.040000,int16,-9.900000e+01,15.0,112.000000,52.669388,18.771257,89.200000,2.164000e+01,89.800000,2.308000e+01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,Distance Sea,Distance_Sea.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,31,1,0.050000,0.050000,float32,-3.402823e+38,0.0,76.643837,24.190548,23.706480,89.150000,2.153000e+01,89.900000,2.308000e+01
796,Distance Sea,Distance_Sea_clip.tif,"PROJCS[""WGS_1984_UTM_Zone_46N"",GEOGCS[""WGS 84""...",12,30,1,5394.264043,5394.264043,float32,-3.402823e+38,0.0,77.234108,23.901064,23.695410,106539.953008,2.389896e+06,171271.121521,2.551723e+06
797,Distance Sea,Distance_Sea_clip2.tif,"PROJCS[""WGS_1984_UTM_Zone_46N"",GEOGCS[""WGS 84""...",12,29,1,5394.264043,5394.264043,float32,-3.402823e+38,0.0,77.234108,23.901064,23.695410,106539.953008,2.395287e+06,171271.121521,2.551720e+06
798,Distance Sea,Distance_Sea_km.tif,"PROJCS[""WGS_1984_UTM_Zone_46N"",GEOGCS[""WGS 84""...",12,30,1,5394.264043,5394.264043,float32,-3.402823e+38,0.0,85.290802,24.488531,24.607689,106539.953008,2.389896e+06,171271.121521,2.551723e+06


In [40]:
# ==========================================================
# Raster Consistency Summary
# ==========================================================

consistency_df = (
    metadata_df
    .groupby("Dataset")
    .agg(
        Total_Files=("File_Name", "count"),
        Unique_CRS=("CRS", "nunique"),
        Unique_Width=("Width", "nunique"),
        Unique_Height=("Height", "nunique"),
        Unique_Res_X=("Resolution_X", "nunique"),
        Unique_Res_Y=("Resolution_Y", "nunique"),
        Unique_Bands=("Bands", "nunique"),
        Unique_DataType=("Data_Type", "nunique"),
        Unique_NoData=("NoData", "nunique"),
    )
    .reset_index()
)

consistency_df

,Dataset,Total_Files,Unique_CRS,Unique_Width,Unique_Height,Unique_Res_X,Unique_Res_Y,Unique_Bands,Unique_DataType,Unique_NoData
0,CCS,72,1,1,1,1,1,1,1,1
1,CDR,72,1,1,1,1,1,1,1,1
2,CHIRPS,72,1,1,1,1,1,1,1,0
3,Distance Sea,5,2,2,3,3,4,1,1,1
4,ERA5,72,1,1,1,1,1,1,2,0
5,GSMaP Gauge,72,1,1,1,1,1,1,1,0
6,GSMaP MVK,72,1,2,2,1,1,1,1,0
7,IMERG,72,1,1,1,1,1,1,1,0
8,LST,72,1,1,1,1,1,1,1,0
9,Land Variable,3,2,2,2,2,2,1,2,0


In [41]:
# ==========================================================
# Add Consistency Status
# ==========================================================

def check_consistency(row):
    checks = [
        row["Unique_CRS"] == 1,
        row["Unique_Width"] == 1,
        row["Unique_Height"] == 1,
        row["Unique_Res_X"] == 1,
        row["Unique_Res_Y"] == 1,
        row["Unique_Bands"] == 1,
        row["Unique_DataType"] == 1,
    ]

    return "CONSISTENT" if all(checks) else "CHECK NEEDED"


consistency_df["Status"] = consistency_df.apply(
    check_consistency,
    axis=1
)

consistency_df

,Dataset,Total_Files,Unique_CRS,Unique_Width,Unique_Height,Unique_Res_X,Unique_Res_Y,Unique_Bands,Unique_DataType,Unique_NoData,Status
0,CCS,72,1,1,1,1,1,1,1,1,CONSISTENT
1,CDR,72,1,1,1,1,1,1,1,1,CONSISTENT
2,CHIRPS,72,1,1,1,1,1,1,1,0,CONSISTENT
3,Distance Sea,5,2,2,3,3,4,1,1,1,CHECK NEEDED
4,ERA5,72,1,1,1,1,1,1,2,0,CHECK NEEDED
5,GSMaP Gauge,72,1,1,1,1,1,1,1,0,CONSISTENT
6,GSMaP MVK,72,1,2,2,1,1,1,1,0,CHECK NEEDED
7,IMERG,72,1,1,1,1,1,1,1,0,CONSISTENT
8,LST,72,1,1,1,1,1,1,1,0,CONSISTENT
9,Land Variable,3,2,2,2,2,2,1,2,0,CHECK NEEDED
